In [0]:
%run /Users/nethumgimsara605@gmail.com/ecommerce-lakehouse-databricks-repo/01-ingestion/setup-storage-connection

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, count, avg, max as spark_max, min as spark_min

In [0]:
silver_orders = spark.read.format("delta").load("abfss://silver@ecommercelakehouse01.dfs.core.windows.net/orders/")
silver_customers = spark.read.format("delta").load("abfss://silver@ecommercelakehouse01.dfs.core.windows.net/customers/").filter("is_current = true")

print(f"Orders: {silver_orders.count()}")
print(f"Customers: {silver_customers.count()}")

In [0]:
orders_with_customer = silver_orders.join(
    silver_customers.select("customer_id", "country", "signup_date"),
    on="customer_id",
    how="inner"

)

print(f"Joined rows: {orders_with_customer.count()}")

In [0]:
gold_customer_ltv = orders_with_customer.groupBy("customer_id", "country").agg(
    spark_sum("total_amount").alias("lifetime_value"),
    count("order_id").alias("total_orders"),
    avg("total_amount").alias("avg_order_value"),
    spark_min("order_timestamp").alias("first_order_date"),
    spark_max("order_timestamp").alias("last_order_date")
)

gold_customer_ltv.orderBy(col("lifetime_value").desc()).show(10)

In [0]:
gold_customer_ltv.write.format("delta").mode("overwrite").save("abfss://gold@ecommercelakehouse01.dfs.core.windows.net/customer_ltv/")

In [0]:
gold_ltv_check = spark.read.format("delta").load("abfss://gold@ecommercelakehouse01.dfs.core.windows.net/customer_ltv/")
print(f"Total rows: {gold_ltv_check.count()}")
gold_ltv_check.orderBy(col("lifetime_value").desc()).show(10)